# Phutball Transformer - GPU/TPU Training

Runtime → A100 GPU or TPU v6e → Run all

In [ ]:
# Detect environment
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ or 'google.colab' in str(globals())
print(f"Running in Colab: {IN_COLAB}")

Running in Colab: True


In [ ]:
# Install dependencies
if IN_COLAB:
    import subprocess
    # Check if TPU is available
    try:
        import jax
        if 'TPU' in str(jax.devices()):
            !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
        else:
            raise Exception("No TPU")
    except:
        # GPU or CPU - install CUDA version
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax wandb mctx

In [ ]:
# Verify devices
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

DEVICE_COUNT = jax.device_count()
DEVICE_TYPE = str(jax.devices()[0]).split(':')[0] if jax.devices() else 'cpu'
print(f"\nUsing {DEVICE_COUNT}x {DEVICE_TYPE}")

JAX version: 0.7.2
Devices: [CudaDevice(id=0)]
Device count: 1

Using 1x cuda


In [ ]:
# Mount Google Drive for checkpoints
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Board size (defined early so checkpoint path uses it)
# Jacob's Ladder: train on small boards, escalate on ELO plateau
BOARD_LADDER_SIZES = ((13, 7), (15, 9), (17, 11), (19, 13), (21, 15))
ROWS, COLS = BOARD_LADDER_SIZES[0]  # Start at first rung

# Architecture params for checkpoint path
D_MODEL = 512
N_LAYERS = 12

if IN_COLAB:
    CHECKPOINT_BASE = f"/content/drive/MyDrive/phutball_checkpoints/transformer_{ROWS}x{COLS}_d{D_MODEL}_l{N_LAYERS}"
else:
    CHECKPOINT_BASE = f"./checkpoints_transformer_{ROWS}x{COLS}_d{D_MODEL}_l{N_LAYERS}"

os.makedirs(CHECKPOINT_BASE, exist_ok=True)
print(f"Checkpoints: {CHECKPOINT_BASE}")

In [ ]:
# Wandb login (optional)
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login()

# === WANDB RESUME ===
# Set to a run ID to resume logging to that run.
# Leave as None to auto-detect from checkpoint dir, or start fresh.
WANDB_RUN_ID = None  # e.g. "abc123def"

# === FRESH START vs RESUME ===
# Set FRESH_START = True when restarting training from scratch.
# This clears the saved wandb run ID so a new run is created.
# Set FRESH_START = False (default) to resume a crashed/stopped run.
FRESH_START = False

if FRESH_START:
    wandb_id_path = os.path.join(CHECKPOINT_BASE, "wandb_run_id.txt")
    if os.path.exists(wandb_id_path):
        old_id = open(wandb_id_path).read().strip()
        os.remove(wandb_id_path)
        print(f"Cleared old wandb run ID: {old_id}")
    WANDB_RUN_ID = None
    print("FRESH START: new wandb run will be created")
else:
    print("Resume mode: will reuse existing wandb run if found")

In [ ]:
# === MERGE PREVIOUS WANDB RUNS (run once) ===
# Merges multiple wandb runs into a single run so you get
# continuous charts. Run this ONCE, then set the resulting
# run ID as WANDB_RUN_ID above for future training.

MERGE_RUNS = False  # <-- Set True to merge

if MERGE_RUNS and USE_WANDB:
    import wandb
    api = wandb.Api()

    # Put your run IDs here in chronological order
    OLD_RUN_IDS = [
        "REPLACE_WITH_RUN1_ID",  # e.g. iter 0-129
        "REPLACE_WITH_RUN2_ID",  # e.g. iter 130-229
        "REPLACE_WITH_RUN3_ID",  # e.g. iter 230-351
    ]
    WANDB_ENTITY = None  # your wandb username, or None for default

    merged_run = wandb.init(
        project=WANDB_PROJECT,
        name=f"transformer_{ROWS}x{COLS}_curriculum_v1_merged",
    )

    total_steps = 0
    for run_id in OLD_RUN_IDS:
        entity = WANDB_ENTITY or api.default_entity
        old_run = api.run(f"{entity}/{WANDB_PROJECT}/{run_id}")
        print(f"Merging {run_id} ({old_run.name})...")

        rows = list(old_run.scan_history())
        print(f"  {len(rows)} steps")

        for row in rows:
            # Filter wandb internal keys
            clean = {k: v for k, v in row.items()
                     if not k.startswith("_") and v is not None}
            if clean:
                # Use iteration as step if available, else sequential
                step = int(row.get("iteration", total_steps))
                wandb.log(clean, step=step)
                total_steps += 1

    print(f"\nMerged {total_steps} steps into run: {merged_run.id}")
    print(f"URL: {merged_run.url}")

    # Save as the run ID for future resumes
    id_path = os.path.join(CHECKPOINT_BASE, "wandb_run_id.txt")
    with open(id_path, "w") as f:
        f.write(merged_run.id)
    print(f"Saved to {id_path} — future runs will auto-resume this.")

    merged_run.finish()
    print("Done! Set MERGE_RUNS = False and re-run the notebook.")


In [ ]:
# Clone/update repo
REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3

Already up to date.
/content/phutball-jax
b2f1f8d (HEAD -> main, origin/main, origin/HEAD) Support resuming wandb runs across training restarts
e297a21 Rewrite profiling notebook to break down per-MCTS-move internals
2c333d7 Add transformer support to profiling scripts and Colab notebook


In [ ]:
# === TRAINING CONFIG ===
# ROWS, COLS, D_MODEL, N_LAYERS, BOARD_LADDER_SIZES defined in cell-4

BOARD_LADDER_ENABLED = True
POS_ENCODING = "goal_distance"  # "normalized" or "goal_distance"

# Self-play (64 for A100 40GB, increase to 128 for 80GB)
BATCH_SIZE_GAMES = 64
NUM_SIMULATIONS = 32
GAMES_PER_ITER = 256

# Training
BATCH_SIZE_TRAIN = 256
LEARNING_RATE = 3e-4         # was 1e-3
WEIGHT_DECAY = 1e-4
TRAIN_STEPS_PER_ITER = 1000  # was 200
BUFFER_SIZE = 1_000_000
MIN_BUFFER_SIZE = 10_000     # was 500 (avoid oversampling early)

# Cosine LR schedule (ramps up then decays smoothly)
# Schedule spans all iterations; board bumps don't reset it,
# so later rungs naturally train at lower LR (fine-tuning regime).
COSINE_LR_ENABLED = True
LR_END = 1e-5
LR_WARMUP_ITERS = 10

# Draw value: 0.0 = standard (draw is neutral)
DRAW_VALUE = 0.0

# Temperature schedule
TEMP_THRESHOLD = 60          # was 30 (anneal over more of the game)
TEMP_FINAL = 0.1

# Curriculum (old N-jump style - disabled)
CURRICULUM_ENABLED = False

# ELO evaluation: evaluate vs league pool of past checkpoints
ELO_EVAL_ENABLED = True
ELO_EVAL_GAMES_PER_PERSPECTIVE = 10  # 10 games/perspective x 5 opponents = 100 games
ELO_EVAL_MAX_OPPONENTS = 5
ELO_EVAL_NUM_SIMULATIONS = 32

# ELO-based sim escalation: bump sims when ELO plateaus
ELO_ESCALATION_ENABLED = True
ELO_MIN_IMPROVEMENT = 20.0
ELO_STAGNATION_PATIENCE = 5
ELO_SIM_TIERS = (32, 48, 64)

# Iterations — large enough to climb the full ladder without hitting the cap
NUM_ITERATIONS = 10_000
CHECKPOINT_EVERY = 10

# Notifications (ntfy.sh - install app and subscribe to topic)
NTFY_TOPIC = "phutball-transformer"  # Set to None to disable
HEARTBEAT_MINUTES = 30

WANDB_PROJECT = "phutball-transformer-13x9"

print(f"Board: {ROWS}x{COLS} (ladder: {BOARD_LADDER_SIZES})")
print(f"Transformer: d_model={D_MODEL}, n_layers={N_LAYERS}, pos_encoding={POS_ENCODING}")
print(f"ELO eval: {ELO_EVAL_ENABLED} ({ELO_EVAL_GAMES_PER_PERSPECTIVE} games/perspective, {ELO_EVAL_MAX_OPPONENTS} opponents)")
print(f"ELO escalation: {ELO_ESCALATION_ENABLED} (tiers={ELO_SIM_TIERS}, patience={ELO_STAGNATION_PATIENCE})")
print(f"Board ladder: {BOARD_LADDER_ENABLED} ({len(BOARD_LADDER_SIZES)} rungs)")
print(f"Training: {NUM_ITERATIONS} iterations (cosine LR spans full run)")
print(f"Heartbeat: every {HEARTBEAT_MINUTES} min to ntfy.sh/{NTFY_TOPIC}")
# Auto-load wandb run ID from checkpoint dir
WANDB_ID_FILE = os.path.join(CHECKPOINT_BASE, "wandb_run_id.txt")
if WANDB_RUN_ID is None and os.path.exists(WANDB_ID_FILE):
    with open(WANDB_ID_FILE) as f:
        WANDB_RUN_ID = f.read().strip()
    print(f"Auto-loaded wandb run ID: {WANDB_RUN_ID}")
elif WANDB_RUN_ID:
    print(f"Using manual wandb run ID: {WANDB_RUN_ID}")
else:
    print("Fresh wandb run (ID will be saved for future resumes)")

In [ ]:
# Imports
import numpy as np
import time

from phutball_env_jax import (
    PhutballState, EnvConfig, reset, step, get_legal_actions,
    state_to_network_input, render_board
)
from network import PhutballTransformer, create_transformer_network
from train_batched import TransformerTrainer, TrainConfig

print("Imports OK")

Imports OK


In [ ]:
# Create config
config = TrainConfig(
    rows=ROWS,
    cols=COLS,

    # Transformer uses these as d_model and n_layers
    num_channels=D_MODEL,
    num_res_blocks=N_LAYERS,
    pos_encoding=POS_ENCODING,

    # Self-play
    batch_size_games=BATCH_SIZE_GAMES,
    num_simulations=NUM_SIMULATIONS,
    games_per_iteration=GAMES_PER_ITER,
    temp_threshold=TEMP_THRESHOLD,
    temp_final=TEMP_FINAL,

    # Training
    batch_size_train=BATCH_SIZE_TRAIN,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    train_steps_per_iteration=TRAIN_STEPS_PER_ITER,
    cosine_lr_enabled=COSINE_LR_ENABLED,
    lr_end=LR_END,
    lr_warmup_iters=LR_WARMUP_ITERS,
    draw_value=DRAW_VALUE,
    buffer_size=BUFFER_SIZE,
    min_buffer_size=MIN_BUFFER_SIZE,

    # Old curriculum (N-jump states) - disabled
    curriculum_enabled=CURRICULUM_ENABLED,

    # ELO evaluation
    elo_eval_enabled=ELO_EVAL_ENABLED,
    elo_eval_games_per_perspective=ELO_EVAL_GAMES_PER_PERSPECTIVE,
    elo_eval_max_opponents=ELO_EVAL_MAX_OPPONENTS,
    elo_eval_num_simulations=ELO_EVAL_NUM_SIMULATIONS,

    # ELO-based sim escalation
    elo_escalation_enabled=ELO_ESCALATION_ENABLED,
    elo_min_improvement=ELO_MIN_IMPROVEMENT,
    elo_stagnation_patience=ELO_STAGNATION_PATIENCE,
    elo_sim_tiers=ELO_SIM_TIERS,

    # Board size escalation (Jacob's Ladder)
    board_ladder_enabled=BOARD_LADDER_ENABLED,
    board_ladder_sizes=BOARD_LADDER_SIZES,

    # Iterations
    num_iterations=NUM_ITERATIONS,

    # Checkpointing
    checkpoint_dir=CHECKPOINT_BASE,
    checkpoint_every=CHECKPOINT_EVERY,

    # Notifications
    ntfy_topic=NTFY_TOPIC,
    heartbeat_minutes=HEARTBEAT_MINUTES,

    # Wandb
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=f"transformer_{ROWS}x{COLS}_d{D_MODEL}_l{N_LAYERS}",
    wandb_run_id=WANDB_RUN_ID,
)

print("Config created")
print(f"Self-play: pure self-play (ELO eval replaces phase curriculum)")
print(f"ELO eval: {config.elo_eval_games_per_perspective} games/perspective, "
      f"max {config.elo_eval_max_opponents} opponents, {config.elo_eval_num_simulations} sims")
print(f"Sim escalation: tiers={config.elo_sim_tiers}, patience={config.elo_stagnation_patience}")
print(f"Board ladder: {config.board_ladder_sizes} (enabled={config.board_ladder_enabled})")

In [ ]:
# Create trainer
trainer = TransformerTrainer(config)

# Count parameters
param_count = sum(x.size for x in jax.tree_util.tree_leaves(trainer.params))
print(f"Transformer parameters: {param_count:,}")

[wandb] Run ID: e47intt4 (saved to /content/drive/MyDrive/phutball_checkpoints/transformer_13x9_d512_l12/wandb_run_id.txt)
Transformer parameters: 25,273,476


In [ ]:
# Train!
trainer.train()

Loaded checkpoint from iteration 99 (buffer: 1000000 examples)
  [League] Loading 10 checkpoints into pool...
  [League] Pool initialized with 10 checkpoints: [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]
Resuming from iteration 100
Transformer Training for Phutball (Batched)
Board: 13x9
Network: d_model=512, n_layers=12
Self-play: 256 games/iter (batch=64)
Training: 200 steps/iter (batch=256)
Temperature: 1.0 -> 0.1 after 30 moves
Devices: [CudaDevice(id=0)]

[ntfy] Heartbeat sent: iter 100
Iteration 101/500
----------------------------------------
  Self-play: 256 games (0.5 g/s), 18535 examples | P1:86 P2:122 D:48 (41%/59%)
    avg moves: 72.4, placements: 67.8, jumps: 3.4, jump_len: 1.57, removed: 3.43, adj_conv: 9.2%
  Training: 200 steps (2.4 steps/sec) | lr: 1e-03 | p_loss: 4.1890, v_loss: 0.7015, policy_kl: 0.0733, entropy: 4.189, v_pred: -0.001±0.059
  Iteration time: 550.5s | Buffer: 1000000 examples

Iteration 102/500
----------------------------------------
  Self-play: 256 games

## Evaluation

In [ ]:
# Evaluate vs random
if trainer:
    print("\nFinal evaluation vs random:")
    win_rate, stats = trainer.evaluate_vs_random_batched()
    print(f"Win rate: {win_rate:.1%}")

NameError: name 'trainer' is not defined

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

if trainer and trainer.metrics_history:
    iters = [m['iteration'] for m in trainer.metrics_history]
    p_loss = [m['policy_loss'] for m in trainer.metrics_history]
    v_loss = [m['value_loss'] for m in trainer.metrics_history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(iters, p_loss)
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Policy Loss')
    axes[0].set_title('Policy Loss')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, v_loss)
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Value Loss')
    axes[1].set_title('Value Loss')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot")

In [ ]:
# List checkpoints
import glob

print("Available checkpoints:")
ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_BASE, "*.pkl")))
print(f"Total: {len(ckpts)}")
for c in ckpts[-5:]:
    print(f"  - {os.path.basename(c)}")

In [ ]:
# Delete checkpoints after a certain iteration
# Set DELETE_AFTER to the last checkpoint you want to KEEP (e.g., 50 keeps 0-50, deletes 51+)
# Set to None to skip deletion entirely

DELETE_AFTER = 50  # Keep checkpoints 0-50, delete 51+

import glob
import re

ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_BASE, "*.pkl")))
print(f"Found {len(ckpts)} checkpoints:")

to_delete = []
to_keep = []

for c in ckpts:
    basename = os.path.basename(c)
    # Extract iteration number from checkpoint_NNNNNN.pkl
    match = re.search(r'checkpoint_(\d+)\.pkl', basename)
    if match:
        iter_num = int(match.group(1))
        if DELETE_AFTER is not None and iter_num > DELETE_AFTER:
            to_delete.append(c)
            print(f"  [DELETE] {basename} (iter {iter_num})")
        else:
            to_keep.append(c)
            print(f"  [KEEP]   {basename} (iter {iter_num})")
    else:
        to_keep.append(c)
        print(f"  [KEEP]   {basename} (unknown iter)")

if DELETE_AFTER is None:
    print("\nDELETE_AFTER is None - skipping deletion")
elif len(to_delete) == 0:
    print(f"\nNo checkpoints to delete (all <= {DELETE_AFTER})")
else:
    print(f"\nDeleting {len(to_delete)} checkpoints (keeping {len(to_keep)})...")
    for c in to_delete:
        os.remove(c)
    print(f"Deleted {len(to_delete)} checkpoints")